# Bayesian Personalized Ranking Matrix Factorization (BPR-MF)

In [4]:
# Libraries
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import random
import time

In [5]:
# Load datasets
ratings_df = pd.read_csv("user_ratings_cleaned.csv")
games_df   = pd.read_csv("games_cleaned.csv")

# BPR focuses on whether an interaction happened, not the specific rating value
interactions = ratings_df[["Username", "BGGId"]].drop_duplicates()

# Create mappings
all_users = interactions["Username"].unique()
all_items = interactions["BGGId"].unique()

user2idx = {u: i for i, u in enumerate(all_users)}
item2idx = {g: i for i, g in enumerate(all_items)}
idx2user = {i: u for u, i in user2idx.items()}
idx2item = {i: g for g, i in item2idx.items()}

n_users = len(all_users)
n_items = len(all_items)

interactions["user_idx"] = interactions["Username"].map(user2idx)
interactions["item_idx"] = interactions["BGGId"].map(item2idx)

user_positives = interactions.groupby("user_idx")["item_idx"].apply(set).to_dict()
print(f"Dataset stats: \n{n_users:,} users \n{n_items:,} items \n{len(interactions):,} interactions")

Dataset stats: 
201,558 users 
10,011 items 
1,047,436 interactions


In [6]:
train_positives = {}
test_pairs = []

for user_idx, pos_set in user_positives.items():
    pos_list = list(pos_set)
    if len(pos_list) < 2:
        train_positives[user_idx] = pos_set
        continue

    held_out = random.choice(pos_list)
    test_pairs.append((user_idx, held_out))
    train_positives[user_idx] = pos_set - {held_out}

print(f"Training on {sum(len(v) for v in train_positives.values()):,} items")

Training on 924,363 items


# BPR-MF Model

In [7]:
class BPR_MF:
    def __init__(self, n_users, n_items, n_factors=32, lr=0.01, reg=0.001):
        self.n_users = n_users
        self.n_items = n_items
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg

        # Initialize user and item latent vectors
        self.U = np.random.normal(0, 0.01, (n_users, n_factors))
        self.V = np.random.normal(0, 0.01, (n_items, n_factors))

    def score(self, user_idx, item_idx):
        return self.U[user_idx] @ self.V[item_idx]

    def bpr_update(self, user_idx, pos_idx, neg_idx):
        u = self.U[user_idx]
        v_pos = self.V[pos_idx]
        v_neg = self.V[neg_idx]

        # Predicted difference
        x_uij = u @ v_pos - u @ v_neg

        # Gradient calculation
        sigmoid = 1.0 / (1.0 + np.exp(-x_uij))
        grad_coeff = 1.0 - sigmoid

        # Update with L2 regularization
        self.U[user_idx] += self.lr * (grad_coeff * (v_pos - v_neg) - self.reg * u)
        self.V[pos_idx]  += self.lr * (grad_coeff * u - self.reg * v_pos)
        self.V[neg_idx]  += self.lr * (-grad_coeff * u - self.reg * v_neg)

    def train(self, train_positives, n_epochs=10, n_samples_per_epoch=200000):
        user_list = [(u, list(pos)) for u, pos in train_positives.items() if len(pos) > 0]

        for epoch in range(1, n_epochs + 1):
            start = time.time()
            total_loss = 0.0

            for _ in tqdm(range(n_samples_per_epoch), desc=f"Epoch {epoch}/{n_epochs}", leave=True):
                # Sample triple
                u_idx, pos_items = random.choice(user_list)
                p_idx = random.choice(pos_items)

                n_idx = random.randint(0, self.n_items - 1)
                while n_idx in train_positives.get(u_idx, set()):
                    n_idx = random.randint(0, self.n_items - 1)

                # Forward for loss logging
                x_uij = self.score(u_idx, p_idx) - self.score(u_idx, n_idx)
                total_loss += -np.log(1e-10 + 1.0 / (1.0 + np.exp(-x_uij)))

                # Backprop
                self.bpr_update(u_idx, p_idx, n_idx)

            print(f"Epoch {epoch} | Loss: {total_loss/n_samples_per_epoch:.4f} | Time: {time.time()-start:.1f}s")

    def recommend(self, user_idx, train_positives, top_n=10):
        scores = self.V @ self.U[user_idx]
        # Filter items the user has already seen
        seen = list(train_positives.get(user_idx, set()))
        scores[seen] = -np.inf
        return np.argsort(scores)[::-1][:top_n]

In [8]:
# Hyperparameters
N_FACTORS = 32
LR = 0.01
REG = 0.001
N_EPOCHS = 10
SAMPLES = 200_000 # Increase this for higher accuracy if you have time

model = BPR_MF(n_users, n_items, N_FACTORS, LR, REG)
model.train(train_positives, n_epochs=N_EPOCHS, n_samples_per_epoch=SAMPLES)

Epoch 1/10:   0%|          | 0/200000 [00:00<?, ?it/s]

Epoch 1 | Loss: 0.6931 | Time: 6.0s


Epoch 2/10:   0%|          | 0/200000 [00:00<?, ?it/s]

Epoch 2 | Loss: 0.6931 | Time: 5.4s


Epoch 3/10:   0%|          | 0/200000 [00:00<?, ?it/s]

...

Epoch 8 | Loss: 0.6487 | Time: 5.4s


Epoch 9/10:   0%|          | 0/200000 [00:00<?, ?it/s]

Epoch 9 | Loss: 0.6122 | Time: 6.0s


Epoch 10/10:   0%|          | 0/200000 [00:00<?, ?it/s]

Epoch 10 | Loss: 0.5824 | Time: 5.5s


# Evaluation
Metric: Hit Rate @ K
For each test (user, held-out item) pair, we ask: Is the held-out item in the model's top-K recommendations?

Hit Rate = (number of hits) / (number of test users)

In [9]:
def evaluate_hit_rate(model, test_pairs, train_positives, k=10):
    hits = 0
    # Sample test users if the set is too large for fast feedback
    sample_test = random.sample(test_pairs, min(2000, len(test_pairs)))

    for u_idx, held_out in tqdm(sample_test, desc=f"HR@{k}"):
        top_k = model.recommend(u_idx, train_positives, top_n=k)
        if held_out in top_k:
            hits += 1
    return hits / len(sample_test)

for k in [10, 20]:
    hr = evaluate_hit_rate(model, test_pairs, train_positives, k=k)
    print(f"Hit Rate @ {k}: {hr:.4%}")

HR@10:   0%|          | 0/2000 [00:00<?, ?it/s]

Hit Rate @ 10: 26.6500%


HR@20:   0%|          | 0/2000 [00:00<?, ?it/s]

Hit Rate @ 20: 28.0500%


# Sample Recommendations

In [12]:
id2name = games_df.set_index("BGGId")["Name"].to_dict()

def show_recs(username, top_n=10):
    if username not in user2idx:
        return "User not found."
    u_idx = user2idx[username]
    top_idxs = model.recommend(u_idx, train_positives, top_n=top_n)

    print(f"Top {top_n} recommendations for user {username}")
    for i, item_idx in enumerate(top_idxs, 1):
        bgg_id = idx2item[item_idx]
        name = id2name.get(bgg_id, f"Unknown ID: {bgg_id}")
        print(f"{i}. {name}")

# Try it with a random user
demo_user = random.choice(all_users)
show_recs(demo_user)

Top 10 recommendations for user krzysrog
1. King of Indecision
2. Jochen der Rochen
3. Ucho Króla
4. Encantados (Second Edition)
5. Pokémon Master Trainer II
6. Sabbat Magica
7. Nonsense Classic
8. Settlement
9. Hoppla Lama
10. The Game of Life: Pirates of the Caribbean
